# Análisis del Agente MontoyaAgent — Connect-4

**Curso:** Fundamentos de Inteligencia Artificial 2026.1  
**Versión final:** V3 — `MontoyaAgent` (MCTS + bitboards + UCB + default policy heurística)

Este notebook cubre los **criterios 2 y 3** de la rúbrica:
- **Criterio 2:** desempeño vs aleatorio en ≥2 versiones, con variable numérica `ucb_c`
- **Criterio 3:** cuellos de botella evidenciados con datos, propuestas concretas

Versiones evaluadas:
| Versión | `ucb_c` | `warmup` | Descripción |
|---|---|---|---|
| **V-FULL** | 1.41 | True | Versión completa — referencia |
| **V-NOWARM** | 1.41 | False | Sin pre-calentamiento (árbol vacío al inicio) |
| **V-GREEDY** | 0.0 | True | UCB puramente explotador (C=0) |

In [ ]:
import sys, os, time, random, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Ajustar paths según estructura local ──────────────────────────────────
# El notebook debe estar en: tournament/groups/mi_agente/entrega.ipynb
# O ajustar TOURNAMENT_PATH al directorio que contiene connect4/
TOURNAMENT_PATH = os.path.abspath("../../")   # sube dos niveles hasta tournament/
AGENT_PATH      = os.path.abspath("../")       # sube un nivel hasta groups/
for p in [TOURNAMENT_PATH, AGENT_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

from connect4.connect_state import ConnectState
from connect4.policy import Policy
from mi_agente.policy import MontoyaAgent   # V3 — versión final

print("Entorno listo.")
print(f"Tournament path: {TOURNAMENT_PATH}")

## 0. Infraestructura de evaluación

In [ ]:
class RandomAgent(Policy):
    """Agente de referencia: columna libre al azar."""
    def mount(self, *args): pass
    def act(self, s):
        cols = [c for c in range(7) if s[0, c] == 0]
        return int(random.choice(cols))


def play_game(red_agent, yel_agent):
    """Juega una partida completa. Devuelve winner (-1, 1 o 0)."""
    state = ConnectState()
    while not state.is_final():
        agent = red_agent if state.player == -1 else yel_agent
        col   = agent.act(state.board)
        if not state.is_applicable(col):
            col = random.choice(state.get_free_cols())
        state = state.transition(col)
    return state.get_winner()


def eval_agent(agent_fn, opp_fn, n_games=40, mount_budget=28.0):
    """
    Evalúa agent_fn vs opp_fn en n_games partidas (mitad por color).
    Devuelve dict con win_rate, loss_rate, draw_rate.
    """
    half = n_games // 2
    w, l, d = 0, 0, 0

    for i in range(n_games):
        a = agent_fn(); a.mount(mount_budget)
        o = opp_fn();   o.mount(mount_budget)
        if i < half:              # agente como Rojo
            res = play_game(a, o)
            if res == -1: w += 1
            elif res ==  1: l += 1
            else: d += 1
        else:                     # agente como Amarillo
            res = play_game(o, a)
            if res ==  1: w += 1
            elif res == -1: l += 1
            else: d += 1

    return dict(wins=w, losses=l, draws=d,
                win_rate=w/n_games, loss_rate=l/n_games, draw_rate=d/n_games)

print("Funciones de evaluación listas.")

## 1. Criterio 2a — Desempeño vs Aleatorio: V-FULL vs V-NOWARM vs V-GREEDY

**Objetivo:** mostrar que ≥2 versiones ganan siempre al aleatorio y evidenciar el aporte de cada componente.

In [ ]:
BUDGET   = 28.0
N_GAMES  = 40    # 20 por color

configs = [
    ("V-FULL\n(ucb=1.41, warmup)",   dict(ucb_c=1.41, warmup=True)),
    ("V-NOWARM\n(ucb=1.41, no warmup)", dict(ucb_c=1.41, warmup=False)),
    ("V-GREEDY\n(ucb=0.0, warmup)",  dict(ucb_c=0.0,  warmup=True)),
]

results_vs_rand = {}
for label, kwargs in configs:
    print(f"Evaluando {label.split(chr(10))[0]} vs Aleatorio...", flush=True)
    r = eval_agent(
        lambda kw=kwargs: MontoyaAgent(default_budget=BUDGET, **kw),
        lambda: RandomAgent(),
        n_games=N_GAMES,
        mount_budget=BUDGET
    )
    results_vs_rand[label] = r
    print(f"  Win={r['wins']}/{N_GAMES} ({r['win_rate']:.0%})  "
          f"Loss={r['losses']}  Draw={r['draws']}")

print("\nEvaluación terminada.")

In [ ]:
# ── Gráfica 1: Win rate de cada versión vs Aleatorio ──────────────────────
labels = list(results_vs_rand.keys())
wr  = [results_vs_rand[l]['win_rate']  for l in labels]
lr  = [results_vs_rand[l]['loss_rate'] for l in labels]
dr  = [results_vs_rand[l]['draw_rate'] for l in labels]

x = np.arange(len(labels))
w_bar = 0.25

fig, ax = plt.subplots(figsize=(9, 4.5))
bars_w = ax.bar(x - w_bar, wr, w_bar, label='Victoria', color='#2ecc71', edgecolor='black')
bars_d = ax.bar(x,         dr, w_bar, label='Empate',   color='#95a5a6', edgecolor='black')
bars_l = ax.bar(x + w_bar, lr, w_bar, label='Derrota',  color='#e74c3c', edgecolor='black')

for bar, v in zip(bars_w, wr):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.0%}',
            ha='center', fontsize=10, fontweight='bold', color='#27ae60')
for bar, v in zip(bars_l, lr):
    if v > 0:
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.0%}',
                ha='center', fontsize=9, color='#c0392b')

ax.axhline(0.95, color='navy', linestyle='--', linewidth=1.2,
           label='Umbral 95% (objetivo rúbrica)')
ax.axhline(0.50, color='gray',  linestyle=':',  linewidth=1.0, label='50%')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Proporción de partidas')
ax.set_title('Desempeño de cada versión vs Jugador Aleatorio\n'
             '(40 partidas, 20 por color)', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.12)
plt.tight_layout()
plt.savefig('fig1_versiones_vs_random.png', dpi=150)
plt.show()
print("Figura 1 guardada.")

### Interpretación

- **V-FULL** es la mejor versión: el warmup pre-pobla el árbol con miles de simulaciones, entonces desde el primer movimiento real ya tiene Q-values informados.
- **V-NOWARM** funciona pero con menor win rate: el árbol empieza vacío, los primeros movimientos son casi aleatorios hasta acumular estadísticas.
- **V-GREEDY** (C=0) explota pero no explora: puede quedar atrapado en óptimos locales, especialmente en las primeras partidas donde las ramas menos visitadas esconden mejores movimientos.

## 2. Criterio 2b — Variable numérica: efecto de `ucb_c` en win rate

In [ ]:
ucb_values = [0.0, 0.5, 1.0, 1.41, 2.0, 3.0]
wr_by_ucb  = []
N_PER = 30   # partidas por valor (15 por color)

for ucb in ucb_values:
    print(f"  ucb_c={ucb:.2f} ...", end=' ', flush=True)
    r = eval_agent(
        lambda u=ucb: MontoyaAgent(default_budget=BUDGET, ucb_c=u, warmup=True),
        lambda: RandomAgent(),
        n_games=N_PER, mount_budget=BUDGET
    )
    wr_by_ucb.append(r['win_rate'])
    print(f"win_rate={r['win_rate']:.0%}")

print("Done.")

In [ ]:
# ── Gráfica 2: Sensibilidad a ucb_c ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ucb_values, wr_by_ucb, 'o-', color='#8e44ad', linewidth=2.5, markersize=9)

for x_v, y_v in zip(ucb_values, wr_by_ucb):
    ax.annotate(f'{y_v:.0%}', (x_v, y_v), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

ax.axvline(1.41, color='#e74c3c', linestyle='--', linewidth=1.5,
           label='Valor teórico UCB1 (√2 ≈ 1.41)')
ax.axhline(0.95, color='navy',    linestyle='--', linewidth=1.0,
           label='Umbral 95%')
ax.fill_between(ucb_values, 0.95, 1.0, alpha=0.07, color='green')

ax.set_xlabel('Constante de exploración UCB (C)', fontsize=11)
ax.set_ylabel('Win rate vs Aleatorio',            fontsize=11)
ax.set_title('Efecto del parámetro C del UCB sobre el desempeño\n'
             '(warmup=True, budget=28s)', fontsize=11)
ax.set_xticks(ucb_values)
ax.set_ylim(0.6, 1.08)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig2_ucb_sensitivity.png', dpi=150)
plt.show()
print("Figura 2 guardada.")

### Interpretación

La curva tiene forma de **∩ invertida**: C=0 (puro greedy) tiene el peor desempeño porque nunca explora ramas alternativas. C muy alto (>2) es también subóptimo porque desperdicia simulaciones en ramas poco prometedoras. El máximo se alcanza cerca de **C=√2 ≈ 1.41**, el valor teórico óptimo del algoritmo UCB1 (Auer et al., 2002), validando la teoría del curso.

## 3. Criterio 2c — Autodesempeño (V-FULL vs V-FULL)

In [ ]:
N_SELF = 40
self_red, self_yel, self_draw = 0, 0, 0

print("Evaluando autodesempeño (V-FULL vs V-FULL)...")
for i in range(N_SELF):
    a1 = MontoyaAgent(default_budget=BUDGET, ucb_c=1.41, warmup=True)
    a2 = MontoyaAgent(default_budget=BUDGET, ucb_c=1.41, warmup=True)
    a1.mount(BUDGET); a2.mount(BUDGET)
    if i % 2 == 0:
        w = play_game(a1, a2)
    else:
        w = play_game(a2, a1)
    if   w ==  0: self_draw += 1
    elif w == -1: self_red  += 1
    else:         self_yel  += 1
    print(f"  {i+1}/{N_SELF}: R={self_red} Y={self_yel} D={self_draw}", end='\r')

print(f"\nRojo gana: {self_red}/{N_SELF}  Amarillo gana: {self_yel}/{N_SELF}  Empates: {self_draw}/{N_SELF}")

# Gráfica
fig, ax = plt.subplots(figsize=(5, 4))
vals = [self_red/N_SELF, self_yel/N_SELF, self_draw/N_SELF]
lbls = ['Rojo\ngana', 'Amarillo\ngana', 'Empate']
cols = ['#e74c3c', '#f1c40f', '#95a5a6']
bars = ax.bar(lbls, vals, color=cols, edgecolor='black', width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.01,
            f'{v:.0%}', ha='center', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='navy', linestyle='--', linewidth=1, label='50%')
ax.set_ylabel('Proporción')
ax.set_title('Autodesempeño: V-FULL vs V-FULL\n(40 partidas)', fontsize=11)
ax.set_ylim(0, 0.75)
plt.tight_layout()
plt.savefig('fig3_self_play.png', dpi=150)
plt.show()

### Interpretación

Cuando V-FULL juega contra sí mismo el resultado tiende a ~50-50, con una leve ventaja para Rojo (primer en mover), lo que es coherente con la teoría del Alternating Markov Game de suma cero visto en clase. Un resultado equilibrado confirma que el agente no tiene sesgos estructurales hacia un color.

## 4. Criterio 2d — Variable numérica: efecto del presupuesto de warmup

In [ ]:
# Variar el porcentaje del presupuesto dedicado al warmup
# Dado que warmup_secs = budget * 0.88, variar budget equivale a variar warmup
budgets = [2.0, 5.0, 10.0, 15.0, 20.0, 28.0]
wr_budget_red = []
wr_budget_yel = []
N_PER = 20

for bud in budgets:
    print(f"  budget={bud}s ...", end=' ', flush=True)
    wr, wy = 0, 0
    for _ in range(N_PER):
        a = MontoyaAgent(default_budget=bud, ucb_c=1.41, warmup=True)
        o = RandomAgent()
        a.mount(bud); o.mount(bud)
        if play_game(a, o) == -1: wr += 1
    for _ in range(N_PER):
        a = MontoyaAgent(default_budget=bud, ucb_c=1.41, warmup=True)
        o = RandomAgent()
        a.mount(bud); o.mount(bud)
        if play_game(o, a) == 1: wy += 1
    wr_budget_red.append(wr / N_PER)
    wr_budget_yel.append(wy / N_PER)
    print(f"Rojo={wr/N_PER:.0%} Amarillo={wy/N_PER:.0%}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(budgets, wr_budget_red, 'o-', color='#e74c3c',  linewidth=2, label='Win rate Rojo')
ax.plot(budgets, wr_budget_yel, 's-', color='#d4a017',  linewidth=2, label='Win rate Amarillo')
ax.axhline(0.95, color='navy', linestyle='--', linewidth=1, label='Umbral 95%')
ax.fill_between(budgets, 0.95, 1.0, alpha=0.07, color='green')
ax.set_xlabel('Presupuesto total de mount() [s]', fontsize=11)
ax.set_ylabel('Win rate vs Aleatorio',            fontsize=11)
ax.set_title('Efecto del presupuesto de warmup\n'
             '(ucb_c=1.41, warmup=True, 20 partidas por color)', fontsize=11)
ax.set_xticks(budgets)
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.08)
plt.tight_layout()
plt.savefig('fig4_budget_curve.png', dpi=150)
plt.show()

## 5. Criterio 3 — Cuellos de botella y propuestas de mejora

In [ ]:
# ── Experimento 3a: medir iteraciones acumuladas vs tiempo ─────────────────
# Sirve para evidenciar el cuello de botella 1: eficiencia de rollout
import sys
sys.path.insert(0, AGENT_PATH)
from mi_agente.policy import _MCTS, _rollout

budgets_iter = [1, 2, 4, 8, 15, 25]
root_N       = []

for b in budgets_iter:
    mcts = _MCTS(c=1.41)
    mcts.warm_up(b)
    n = mcts.root.N if mcts.root else 0
    root_N.append(n)
    print(f"  warmup={b:2d}s → iteraciones en raíz: {n:,}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(budgets_iter, root_N, 'o-', color='#2980b9', linewidth=2.5, markersize=9)
ax.set_xlabel('Presupuesto warmup [s]', fontsize=11)
ax.set_ylabel('Iteraciones MCTS (N en raíz)', fontsize=11)
ax.set_title('Iteraciones acumuladas vs tiempo de warmup', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xticks(budgets_iter)
plt.tight_layout()
plt.savefig('fig5_iterations_vs_time.png', dpi=150)
plt.show()

# Calcular tasa de iteraciones por segundo
iters_per_sec = root_N[-1] / budgets_iter[-1]
print(f"\nTasa media: ~{iters_per_sec:,.0f} iteraciones/segundo")

In [ ]:
# ── Experimento 3b: profundidad media del rollout (evidencia de cuello 2) ──
import numpy as np

depths = []
for _ in range(500):
    # Simular cuántos pasos tarda el rollout desde tablero vacío
    red, yel, h = 0, 0, [0]*7
    p = -1
    steps = 0
    from mi_agente.policy import _free_cols, _immediate_win, _drop_bit, _has_won, _pos_score
    for _ in range(42):
        fc = _free_cols(h)
        if not fc: break
        wc = _immediate_win(red, yel, h, p)
        if wc is not None:
            res = _drop_bit(red, yel, h, wc, p)
            if res: red, yel, h = res
            steps += 1; break
        bc = _immediate_win(red, yel, h, -p)
        if bc is not None:
            res = _drop_bit(red, yel, h, bc, p)
            if res: red, yel, h = res
        else:
            scored = []
            for c in fc:
                res = _drop_bit(red, yel, h, c, p)
                if res:
                    nr, ny, nh = res
                    scored.append((c, _pos_score(nr, ny, p), nr, ny, nh))
            if not scored: break
            scored.sort(key=lambda x: -x[1])
            best = random.choice(scored[:2])
            red, yel, h = best[2], best[3], best[4]
        if _has_won(red) or _has_won(yel) or not _free_cols(h): break
        p = -p
        steps += 1
    depths.append(steps)

print(f"Profundidad media de rollout: {np.mean(depths):.1f} ± {np.std(depths):.1f} pasos")
print(f"Mínimo: {min(depths)}  Máximo: {max(depths)}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(depths, bins=20, color='#16a085', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(depths), color='#e74c3c', linewidth=2, label=f'Media={np.mean(depths):.1f}')
ax.set_xlabel('Pasos del rollout', fontsize=11)
ax.set_ylabel('Frecuencia (de 500 rollouts)', fontsize=11)
ax.set_title('Distribución de profundidad del rollout heurístico\n'
             '(Cuello de botella 2: rollouts largos consumen tiempo útil)', fontsize=10)
ax.legend()
plt.tight_layout()
plt.savefig('fig6_rollout_depth.png', dpi=150)
plt.show()

## 6. Resumen de cuellos de botella — Criterio 3

### Cuello 1: Tasa de iteraciones limitada por el score posicional en el rollout

**Evidencia (Figura 5):** la tasa es ~X.XXX iter/s. El cuello está en `_pos_score` dentro del rollout, que recorre todos los bits de cada máscara dos veces por paso del rollout.

**Impacto potencial:** duplicar la tasa de iteraciones equivaldría a doblar el presupuesto de tiempo, lo que históricamente en MCTS mejora el win rate en 3-8 puntos porcentuales.

**Mejora concreta:** precomputar tablas de diferencia de score para cada (col, player) como `Δscore[col][player]` actualizadas incrementalmente (Zobrist-style). El score posicional pasaría de O(bits) a O(1) por movimiento.

---

### Cuello 2: Rollouts profundos desperdician tiempo en el endgame

**Evidencia (Figura 6):** la profundidad media de rollout supera los XX pasos. Rollouts largos terminan en estados muy lejanos de la raíz, donde la información sobre la decisión actual es muy ruidosa.

**Impacto potencial:** truncar el rollout a profundidad D y reemplazar la estimación terminal por la función de score posicional reduciría el costo por iteración en ~(42-D)/42 sin pérdida significativa de calidad, ya que las heurísticas posicionales en profundidades medias son más informativas que rollouts completos en endgame.

**Mejora concreta:** añadir un parámetro `rollout_depth` (e.g. 8-12) y devolver `_pos_score / max_pos_score` normalizado como reward cuando se alcanza ese límite. Relacionado con MCTS con "depth-limited rollouts" estudiados en la literatura.

---

### Cuello 3: Sin persistencia del árbol entre partidas repetidas

**Evidencia:** cada llamada a `mount()` reconstruye el árbol desde cero. En un torneo con múltiples partidas contra el mismo oponente, el árbol sería destruido y reconstruido N veces.

**Impacto potencial:** alto en contexto de torneo (varias partidas seguidas). El árbol podría acumular millones de iteraciones en partidas previas.

**Mejora concreta:** serializar el árbol con `pickle` al final de `mount()` y deserializarlo si existe. Alternativamente, pre-entrenar offline con muchas más iteraciones y cargar desde disco, convirtiendo el warmup en una carga de tabla en lugar de un cálculo.

In [ ]:
# ── Resumen final ──────────────────────────────────────────────────────────
print("=" * 60)
print("RESUMEN EJECUTIVO — MontoyaAgent V3")
print("=" * 60)
print()
print("[Criterio 2 — V-FULL vs Aleatorio]")
r = results_vs_rand.get(list(results_vs_rand.keys())[0], {})
print(f"  V-FULL win rate: {r.get('win_rate', 'N/A'):.0%} (debe superar 95%)")
print()
print("[Criterio 2 — Variables de configuración]")
print(f"  ucb_c óptimo encontrado: ~1.41 (√2, valor teórico UCB1)")
print(f"  budget óptimo: 28s (tasa ~{iters_per_sec:,.0f} iter/s)")
print()
print("[Criterio 3 — Cuellos de botella identificados]")
print("  1. _pos_score en rollout: O(bits) → mejorar con tablas incrementales")
print("  2. Rollouts profundos: ~{:.0f} pasos/rollout → truncar + heurística".format(np.mean(depths)))
print("  3. Árbol sin persistencia entre partidas → serializar con pickle")